# Urdu ASR Pipeline: Ensemble, Normalization & OOV Correction

## Overview
This notebook implements a full post-processing pipeline on top of raw ASR model outputs for Urdu speech recognition. It covers four stages:

1. **Ensemble construction** — merge per-model predictions into a single aligned DataFrame
2. **Text normalization** — strip diacritics, Arabic/Urdu script variants, punctuation, and non-Urdu characters
3. **Vocabulary building** — deduplicate the Mozilla Common Voice corpus into a clean reference vocabulary
4. **OOV correction** — detect out-of-vocabulary tokens and correct them using bigram language model scoring combined with fuzzy string matching (two strategies: weighted interpolation and Reciprocal Rank Fusion)
5. **Evaluation** — compute WER and CER before and after each correction stage

---
**Inputs:** `asr_results/`, `asr_ensemble/ensemble_raw.tsv`, `corpus/`, `vocab/`  
**Outputs:** `asr_ensemble/ensemble_corrected.tsv`, `asr_evaluation/ensemble_correction_eval.tsv`

## 1. Imports

In [ ]:
import pandas as pd
import os
import numpy as np
import re
from collections import defaultdict, Counter
from rapidfuzz import process, fuzz
from math import log

## 2. Directory Configuration

All project paths are defined here as constants. Update these if the folder structure changes.

In [ ]:
# Root directories for each data category
ASR_RESULTS_DIR    = 'asr_results'     # Per-model raw transcription TSVs
ASR_ENSEMBLE_DIR   = 'asr_ensemble'    # Ensemble input/output TSVs
CORPUS_DIR         = 'corpus'          # Mozilla Common Voice corpus splits
METADATA_DIR       = 'metadata'        # Clip duration metadata
VOCAB_DIR          = 'vocab'           # Deduplicated Urdu vocabulary
ASR_EVALUATION_DIR = 'asr_evaluation'  # WER/CER evaluation outputs

## 3. Ensemble Construction

Load all per-model prediction TSVs from `asr_results/` and merge them into a single wide-format DataFrame. Each file contributes one prediction column named after the model (filename minus the `.tsv` extension). The merge is performed on `path` (audio clip filename) to ensure row alignment across models.

In [ ]:
# List all TSV files in the results directory
files = os.listdir(ASR_RESULTS_DIR)

In [ ]:
# Iteratively load and merge each model's predictions on the 'path' key.
# The first file seeds the DataFrame; subsequent files drop 'ground_truth'
# (already present) and are joined by 'path'.
all_model_ensemble_df = None
for file in files:
    file_path = ASR_RESULTS_DIR + '/' + file
    df = pd.read_csv(file_path, sep='\t')
    if all_model_ensemble_df is None:
        # Rename 'prediction' to the model name (strip .tsv extension)
        df.rename(columns={'prediction': re.sub(r"\.tsv$", '', file)}, inplace=True)
        all_model_ensemble_df = df
        continue
    # Drop duplicate 'ground_truth' column before merging
    df.drop(columns=['ground_truth'], inplace=True)
    df.rename(columns={'prediction': re.sub(r"\.tsv$", '', file)}, inplace=True)
    all_model_ensemble_df = pd.merge(all_model_ensemble_df, df, on='path', how='inner')

In [ ]:
all_model_ensemble_df.head()

## 4. Corpus Loading

Load all Common Voice corpus splits from `corpus/` into a list of DataFrames. These splits (train, dev, test, validated, etc.) are used to build the reference vocabulary.

In [ ]:
# Build file paths for all TSV files in the corpus directory
corpus_file_paths = [f"{CORPUS_DIR}/{file}" for file in os.listdir(CORPUS_DIR)]

# Load each split into a DataFrame list
mozilla_corpus_df_list = [pd.read_csv(p, sep='\t') for p in corpus_file_paths]

## 5. Corpus Column Validation

Not all corpus splits contain the same columns (e.g. `unvalidated_sentences.tsv` only has `sentence`). Build a boolean mask to skip splits that are missing any of the required columns: `path`, `sentence_id`, `sentence`.

In [ ]:
# Required columns for vocabulary construction
urdu_vocab_df_columns = ['path', 'sentence_id', 'sentence']

# Build a mask — True means the split is missing at least one required column
mask_index_array = [False] * len(mozilla_corpus_df_list)
for index in range(len(mozilla_corpus_df_list)):
    for column in urdu_vocab_df_columns:
        if column not in mozilla_corpus_df_list[index].columns:
            mask_index_array[index] = True

print(mask_index_array)
print(os.listdir(CORPUS_DIR))

## 6. Corpus Concatenation

Concatenate all valid splits into a single DataFrame. Splits flagged by the mask (missing required columns) are replaced with empty DataFrames to maintain index alignment.

In [ ]:
# Concatenate valid splits; substitute empty DataFrames for masked (incompatible) splits
urdu_vocab_df = pd.concat(
    [
        mozilla_corpus_df_list[i][urdu_vocab_df_columns]
        if not mask_index_array[i]
        else pd.DataFrame(columns=urdu_vocab_df_columns)
        for i in range(len(mozilla_corpus_df_list))
    ],
    ignore_index=True
)

In [ ]:
# Verify uniqueness across the three key identifiers
print(len(urdu_vocab_df['path'].unique()))
print(len(urdu_vocab_df['sentence_id'].unique()))
print(len(urdu_vocab_df['sentence'].unique()))

## 7. Urdu Text Normalization

A deterministic normalizer that maps all text into a canonical Urdu form before any comparison or scoring. Applied to both ASR predictions and ground-truth sentences.

**Steps applied in order:**

| # | Operation | Detail |
|---|---|---|
| 1 | Strip diacritics | Remove zabar, zer, pesh, tanwin, shadda, sukun (`U+0610–U+061A`, `U+064B–U+065F`, `U+0670`) |
| 2 | Remove tatweel | Strip kashida `U+0640` used for visual stretching |
| 3 | Arabic → Urdu mapping | `ك→ک`, `ه→ہ`, `ى→ی`, `ة→ہ`, `أ→ا`, `إ→ا`, `ي→ی` |
| 4 | Remove invisible chars | Zero-width spaces, directional marks (`U+200B–U+202E`) |
| 5 | Remove all punctuation | Urdu punctuation block + ASCII symbols |
| 6 | Remove Latin & digits | Strip all `[A-Za-z0-9]` |
| 7 | Collapse whitespace | Normalize to single spaces |

In [ ]:
import pandas as pd
import re

def normalize_urdu(text):
    """
    Normalize Urdu text to a canonical form for ASR evaluation.

    Removes diacritics, Arabic script variants, punctuation, Latin
    characters, and digits. Maps Arabic characters to their Urdu
    equivalents. Returns a clean, whitespace-normalized Urdu string.

    Args:
        text (str): Raw Urdu text string.

    Returns:
        str: Normalized Urdu string, or empty string if input is NaN/empty.
    """
    if pd.isna(text) or text == '':
        return ''

    # Step 1: Remove diacritics (zabar, zer, pesh, tanwin, shadda, sukun, etc.)
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F\u0670]', '', text)

    # Step 2: Remove tatweel/kashida (ـ) used for visual letter stretching
    text = text.replace('\u0640', '')

    # Step 3: Normalize Arabic characters to their Urdu equivalents
    char_map = {
        '\u0643': '\u06A9',  # Arabic KAF → Urdu KEHEH (ك → ک)
        '\u0647': '\u06C1',  # Arabic HEH → Urdu HEH GOAL (ه → ہ)
        '\u0649': '\u06CC',  # ALEF MAQSURA → Farsi YEH (ى → ی)
        '\u0629': '\u06C1',  # TEH MARBUTA → Urdu HEH GOAL (ة → ہ)
        '\u0623': '\u0627',  # ALEF+HAMZA ABOVE → plain ALEF (أ → ا)
        '\u0625': '\u0627',  # ALEF+HAMZA BELOW → plain ALEF (إ → ا)
        '\u064A': '\u06CC',  # Arabic YEH → Farsi YEH (ي → ی)
    }
    for src, tgt in char_map.items():
        text = text.replace(src, tgt)

    # Step 4: Remove zero-width and invisible Unicode control characters
    text = re.sub(r'[\u200B-\u200F\u202A-\u202E\u2060-\u206F\uFEFF]', '', text)

    # Step 5: Remove all punctuation — Urdu, extended Arabic, and ASCII
    text = re.sub(r'[،؍؎؏۔؟؛٪٬]', '', text)
    text = re.sub(r'[\u0600-\u060C\u060E-\u061F]', '', text)
    text = re.sub(r'[!@#$%^&*()_+=\[\]{};:\'",.<>/?\\|`~\-–—\'\'\"\"…]', '', text)

    # Step 6: Remove all Latin letters and digits
    text = re.sub(r'[A-Za-z0-9]', '', text)

    # Step 7: Collapse multiple spaces into one and strip leading/trailing
    text = re.sub(r'\s+', ' ', text).strip()

    return text

## 8. Load and Normalize Ensemble

Load `ensemble_raw.tsv` and apply `normalize_urdu` to every text column. Normalized columns are stored with a `_norm` suffix alongside the originals.

In [ ]:
# Load the merged ensemble file produced in Section 3
df = pd.read_csv(f"{ASR_ENSEMBLE_DIR}/ensemble_raw.tsv", sep='\t')

print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
print("\nFirst few rows (raw):")
print(df.head(3))

In [ ]:
# Identify all text columns (every column except the 'path' identifier)
text_columns = list(df.columns)
text_columns.remove('path')

# Apply normalization and store results in new '_norm' suffixed columns
for col in text_columns:
    norm_col = f"{col}_norm"
    print(f"Normalizing {col}...")
    df[norm_col] = df[col].apply(normalize_urdu)

print("\n✅ Normalization complete!")

In [ ]:
# Spot-check: compare raw vs normalized text for a single row
sample_idx = 5
print("="*80)
print(f"SAMPLE ROW {sample_idx + 1}:")
print("="*80)

for col in text_columns:
    print(f"\n{col.upper()}:")
    print(f"  BEFORE: {df.loc[sample_idx, col]}")
    print(f"  AFTER:  {df.loc[sample_idx, col + '_norm']}")

In [ ]:
# Persist the normalized ensemble to disk
output_path = f"{ASR_ENSEMBLE_DIR}/ensemble_normalized.tsv"
df.to_csv(output_path, sep='\t', index=False)

print(f"✅ Saved to: {output_path}")
print(f"\nColumns in output file:")
print(df.columns.tolist())

In [ ]:
# Report average character length before and after normalization per column
print("Normalization Impact:")
print("="*80)

for col in text_columns:
    orig_lengths = df[col].str.len().mean()
    norm_lengths = df[col + '_norm'].str.len().mean()
    reduction = (1 - norm_lengths / orig_lengths) * 100
    print(f"{col:20s}: {orig_lengths:6.1f} → {norm_lengths:6.1f} chars ({reduction:5.1f}% reduction)")

## 9. Reference Vocabulary Construction

Build a deduplicated Urdu vocabulary from the full Common Voice corpus. The deduplication key is the **normalized** sentence — two raw sentences that normalize to the same string are treated as the same entry, keeping only the first encountered raw form.

This produces `vocab/urdu_vocab.tsv` with columns:
- `sentence` — normalized form (used as lookup key)
- `raw_sentence` — original raw form

In [ ]:
# Apply normalization to all corpus sentences
urdu_vocab_df['normalized_sentence'] = urdu_vocab_df['sentence'].apply(normalize_urdu)

In [ ]:
# Extract unique sentence lists for raw and normalized forms
normalized_sentence_list = urdu_vocab_df['normalized_sentence'].unique()
raw_sentence_list        = urdu_vocab_df['sentence'].unique()

In [ ]:
# Compare unique counts — normalization typically reduces the count
# by collapsing Arabic/Urdu script variants of the same sentence
print(len(raw_sentence_list))
print(len(normalized_sentence_list))

In [ ]:
# Calibration dict: maps normalized sentence → first seen raw sentence
# Used to collapse duplicates while preserving the original surface form
calibration_dict = {}

In [ ]:
# Sanity check: confirm empty dict returns 0 for unseen keys
print(calibration_dict.get('hello', 0))

In [ ]:
# Populate the calibration dict — first occurrence of each normalized
# form wins; subsequent duplicates are silently skipped
for sentence in raw_sentence_list:
    norm_sentence = normalize_urdu(sentence)
    if calibration_dict.get(norm_sentence, 0) == 0:
        calibration_dict[norm_sentence] = sentence

In [ ]:
# Convert the calibration dict to a DataFrame
# 'sentence' = normalized key, 'raw_sentence' = original surface form
urdu_vocab_dedup_df = pd.DataFrame(
    list(calibration_dict.items()),
    columns=['sentence', 'raw_sentence']
)

In [ ]:
urdu_vocab_dedup_df.head()

In [ ]:
# Save the deduplicated vocabulary to disk
urdu_vocab_dedup_df.to_csv(f"{VOCAB_DIR}/urdu_vocab.tsv", sep='\t', index=False)

In [ ]:
# Reload and verify row count matches what was written
check_df = pd.read_csv(f"{VOCAB_DIR}/urdu_vocab.tsv", sep='\t')

In [ ]:
# Confirm in-memory and on-disk row counts are identical
print(urdu_vocab_dedup_df.count())
print(check_df.count())

## 10. Bigram Language Model

Build a bigram language model over the deduplicated vocabulary corpus. This model is used to rank OOV correction candidates by their contextual fit.

Two smoothing strategies are implemented:

### 10.1 Kneser-Ney Smoothing

Kneser-Ney is the standard smoothing method for n-gram LMs. It replaces the unigram backoff with a **continuation probability** — the number of distinct contexts a word appears in, rather than its raw frequency.

**Forward probability:**

$$P_{KN}(w_i | w_{i-1}) = \frac{\max(c(w_{i-1}, w_i) - D, 0)}{c(w_{i-1})} + \lambda(w_{i-1}) \cdot P_{cont}(w_i)$$

Where:
- $D = 0.75$ — fixed discount
- $\lambda(w_{i-1}) = \frac{D \cdot |\{w : c(w_{i-1}, w) > 0\}|}{c(w_{i-1})}$ — interpolation weight
- $P_{cont}(w_i) = \frac{|\{v : c(v, w_i) > 0\}|}{|\{(u,v) : c(u,v) > 0\}|}$ — continuation probability

The **reverse** direction (right context) mirrors this formula with bigrams reversed.

### 10.2 Raw Bigram MLE

$$P_{MLE}(w_i | w_{i-1}) = \frac{c(w_{i-1}, w_i)}{c(w_{i-1})}$$

Used as a baseline; no smoothing — returns 0 for unseen bigrams.

In [ ]:
# Load the deduplicated vocabulary and filter out any sentences
# containing Latin characters or digits (non-Urdu content)
bigram_df   = pd.read_csv(f"{VOCAB_DIR}/urdu_vocab.tsv", sep='\t')
print(bigram_df.columns)
print(bigram_df.count())

sentence_list = bigram_df['sentence'].dropna().unique()
print(len(sentence_list))

# Remove sentences that still contain Latin/digit characters
sentence_list = [s for s in sentence_list if not re.search("[A-Za-z0-9]", s)]
print(len(sentence_list))

In [ ]:
# Tokenize each sentence into a list of words
token_list = [sentence.split() for sentence in sentence_list]

In [ ]:
# Kneser-Ney discount factor (standard value)
D = 0.75

# Unigram counts and vocabulary set
unigram_count = Counter()
for words in token_list:
    unigram_count.update(words)
vocab_dict = {key: True for key in unigram_count.keys()}
print(len(vocab_dict))

In [ ]:
# Forward bigram counts: c(w1, w2)
bigram_count = Counter()
for words in token_list:
    bigram_count.update(zip(words, words[1:]))

# Continuation count: number of distinct left contexts for each word w2
continuation_counts = Counter(w2 for (_, w2) in bigram_count)
total_bigram_types  = len(bigram_count)

# Forward completions per context: {w1: set of w2 that follow w1}
completions_per_context = defaultdict(set)
for (w1, w2) in bigram_count:
    completions_per_context[w1].add(w2)

# Reverse bigram counts: c(w2, w1) — used for right-context scoring
reverse_bigram_counts = Counter()
for words in token_list:
    reverse_bigram_counts.update(zip(words[1:], words))

# Reverse completions per context: {w2: set of w1 that precede w2}
reverse_completions_per_context = defaultdict(set)
for (w2, w1) in reverse_bigram_counts:
    reverse_completions_per_context[w2].add(w1)

# Reverse continuation counts and total reverse bigram types
reverse_continuation_counts = Counter(w1 for (_, w1) in reverse_bigram_counts)
total_reverse_bigram_types  = len(reverse_bigram_counts)

In [ ]:
def p_kneser_ney(word, context, reverse=False, name=False):
    """
    Compute Kneser-Ney smoothed bigram probability P(word | context).

    Supports both forward P(w_i | w_{i-1}) and reverse P(w_i | w_{i+1})
    directions for left and right context scoring respectively.

    Args:
        word (str):     The word whose probability is being estimated.
        context (str):  The conditioning context word.
        reverse (bool): If True, use reverse bigram counts for right-context scoring.
        name (bool):    If True, return the function name string (used for logging).

    Returns:
        float: Kneser-Ney smoothed probability, or the continuation probability
               as backoff when the context has zero count.
    """
    if name:
        return 'p_kneser_ney'

    if reverse:
        c_vw        = reverse_bigram_counts[(context, word)]
        c_v         = unigram_count[context]
        p_cont      = reverse_continuation_counts[word] / total_reverse_bigram_types
        completions = reverse_completions_per_context[context]
    else:
        c_vw        = bigram_count[(context, word)]
        c_v         = unigram_count[context]
        p_cont      = continuation_counts[word] / total_bigram_types
        completions = completions_per_context[context]

    # Back off to continuation probability when context is unseen
    if c_v == 0:
        return p_cont

    discounted = max(c_vw - D, 0) / c_v
    lam        = D * len(completions) / c_v
    return discounted + lam * p_cont

In [ ]:
def bigram_llm(word, context, reverse=False, name=False):
    """
    Compute raw MLE bigram probability P(word | context).

    No smoothing — returns 0 for unseen bigrams. Used as a baseline
    to compare against Kneser-Ney smoothed scoring.

    Args:
        word (str):     The word whose probability is being estimated.
        context (str):  The conditioning context word.
        reverse (bool): If True, use reverse bigram counts.
        name (bool):    If True, return the function name string.

    Returns:
        float: MLE bigram probability. Zero for unseen bigrams.
    """
    if name:
        return 'bigram_llm'
    if reverse:
        return reverse_bigram_counts[(context, word)] / unigram_count[context]
    return bigram_count[(context, word)] / unigram_count[context]

## 11. Edit Distance (Levenshtein)

Custom implementation of the Levenshtein edit distance algorithm using dynamic programming. Used for both WER/CER scoring and as a similarity signal in OOV correction.

$$\text{lev}(s_1, s_2) = dp[|s_1|][|s_2|]$$

Where $dp[i][j]$ is the minimum number of single-character edits (insert, delete, substitute) to transform $s_1[0..i]$ into $s_2[0..j]$.

Works on both strings (character-level) and lists of tokens (word-level).

In [ ]:
def levenshtein(s1: str | list, s2: str | list, dp_matrix=False) -> float | list:
    """
    Compute the Levenshtein edit distance between two sequences.

    Accepts both strings (character-level distance) and lists of tokens
    (word-level distance), enabling reuse for both CER and WER computation.

    Args:
        s1 (str | list):      Reference sequence.
        s2 (str | list):      Hypothesis sequence.
        dp_matrix (bool):     If True, return the full DP matrix instead of
                              the final scalar distance (used by sclite_align).

    Returns:
        float | list: Edit distance integer, or the full DP table if dp_matrix=True.
    """
    s1_len, s2_len = len(s1), len(s2)

    # Initialize DP table: dp[i][0] = i (delete all of s1), dp[0][j] = j (insert all of s2)
    dp = [
        [i if j == 0 else j if i == 0 else 0
         for j in range(s2_len + 1)]
        for i in range(s1_len + 1)
    ]

    # Fill DP table using the standard recurrence
    for i in range(1, s1_len + 1):
        for j in range(1, s2_len + 1):
            dp[i][j] = min(
                dp[i - 1][j] + 1,                                    # deletion
                dp[i][j - 1] + 1,                                    # insertion
                dp[i - 1][j - 1] + (1 if s1[i-1] != s2[j-1] else 0) # substitution
            )

    return dp[s1_len][s2_len] if not dp_matrix else dp

## 12. Fuzzy Vocabulary Matching

For a given OOV token, retrieve the top-N most similar in-vocabulary words by normalized Levenshtein similarity using `rapidfuzz` for efficient search over the full vocabulary.

In [ ]:
from rapidfuzz.distance import Levenshtein

# Flat list of all in-vocabulary words for fuzzy search
vocab_list = list(vocab_dict.keys())

def fuzzy_match(oov_token, top_n=10):
    """
    Retrieve the top-N in-vocabulary words most similar to an OOV token.

    Uses rapidfuzz's normalized Levenshtein similarity for efficient
    candidate retrieval over the full vocabulary list.

    Args:
        oov_token (str): The out-of-vocabulary token to match.
        top_n (int):     Number of top candidates to return.

    Returns:
        list[tuple[str, float]]: List of (word, similarity_score) pairs,
                                 sorted by similarity descending.
    """
    results = process.extract(
        oov_token,
        vocab_list,
        scorer=Levenshtein.normalized_similarity,
        limit=top_n
    )
    return [(word, score) for word, score, _ in results]

## 13. OOV Correction Strategies

Two correction strategies are implemented. Both use a sliding bigram window over the sentence. For each OOV token, correction candidates are retrieved via `fuzzy_match` and then re-ranked using bigram context.

### Strategy 1: Weighted Interpolation (`resolve_oov_robust`)

Combines the fuzzy similarity score $p_{fuzzy}$ with the bigram LM score $p_{LM}$ using a fixed interpolation weight $\lambda$:

$$\text{score}(w) = \lambda \cdot P_{LM}(w \mid \text{context}) + (1 - \lambda) \cdot p_{fuzzy}(w)$$

### Strategy 2: Reciprocal Rank Fusion (`resolve_oov_rrf`)

Combines fuzzy rank $r_{fuzzy}$ and LM rank $r_{LM}$ using RRF — a rank aggregation method robust to score scale differences:

$$\text{RRF}(w) = \frac{1}{k + r_{fuzzy}(w)} + \frac{1}{k + r_{LM}(w)}$$

Where $k = 60$ is a smoothing constant that reduces the impact of high-ranked outliers.

**Context direction:**
- OOV token has an **in-vocab right neighbour** → use forward bigram $P(w \mid w_{left})$
- OOV token has an **in-vocab left neighbour** → use reverse bigram $P(w \mid w_{right})$
- Both neighbours OOV → fall back to pure fuzzy match on both

In [ ]:
def resolve_oov_robust(current_word, next_word, top_n=5, top_fuzz=20, lam=0.98, bigram_func=p_kneser_ney):
    """
    Resolve OOV tokens using weighted interpolation of fuzzy similarity and bigram LM score.

    Args:
        current_word (str):  The token at position i.
        next_word (str|None): The token at position i+1, or None if at sentence end.
        top_n (int):         Number of correction candidates to return.
        top_fuzz (int):      Fuzzy match candidate pool size before re-ranking.
        lam (float):         Interpolation weight for LM score (1-lam for fuzzy).
        bigram_func:         Bigram scoring function (p_kneser_ney or bigram_llm).

    Returns:
        dict: {
            'current_word_oov': list of (candidate, score) for current_word if OOV,
            'next_word_oov':    list of (candidate, score) for next_word if OOV,
            'type':             correction strategy used
        }
    """
    current_word_oov = not vocab_dict.get(current_word, False)
    next_word_oov    = not vocab_dict.get(next_word, False)

    if current_word_oov and next_word is None:
        # End of sentence — pure fuzzy fallback, no context available
        return {'current_word_oov': fuzzy_match(current_word, top_fuzz)[:top_n], 'next_word_oov': [], 'type': 'fuzzy-match'}

    elif current_word_oov and next_word_oov:
        # Both OOV — no reliable context, apply fuzzy on both independently
        return {'current_word_oov': fuzzy_match(current_word, top_fuzz)[:top_n], 'next_word_oov': fuzzy_match(next_word, top_fuzz), 'type': 'fuzzy_match'}

    elif not current_word_oov and next_word_oov:
        # current is in-vocab → use it as forward context for next_word correction
        candidates = fuzzy_match(next_word, top_fuzz)
        scored = [(w, (lam * bigram_func(w, current_word) + (1 - lam) * p) if p < 0.999 else p) for w, p in candidates]
        scored = sorted(scored, key=lambda x: x[1], reverse=True)
        return {'current_word_oov': [], 'next_word_oov': scored[:top_n], 'type': bigram_func(None, None, None, True)}

    elif current_word_oov and not next_word_oov:
        # next is in-vocab → use it as reverse context for current_word correction
        candidates = fuzzy_match(current_word, top_fuzz)
        scored = [(w, (lam * bigram_func(w, next_word, reverse=True) + (1 - lam) * p) if p < 0.999 else p) for w, p in candidates]
        scored = sorted(scored, key=lambda x: x[1], reverse=True)
        return {'current_word_oov': scored[:top_n], 'next_word_oov': [], 'type': bigram_func(None, None, None, True)}

    else:
        # Both in-vocab — no correction needed
        return {'current_word_oov': [], 'next_word_oov': [], 'type': 'empty'}

In [ ]:
def resolve_oov_rrf(current_word, next_word, top_n=5, top_fuzz=20, k=60, bigram_func=p_kneser_ney):
    """
    Resolve OOV tokens using Reciprocal Rank Fusion of fuzzy and bigram LM rankings.

    Combines fuzzy similarity rank and bigram LM rank via RRF score:
        RRF(w) = 1/(k + rank_fuzzy(w)) + 1/(k + rank_lm(w))

    Args:
        current_word (str):  The token at position i.
        next_word (str|None): The token at position i+1, or None if at sentence end.
        top_n (int):         Number of correction candidates to return.
        top_fuzz (int):      Fuzzy match candidate pool size before re-ranking.
        k (int):             RRF smoothing constant (default 60).
        bigram_func:         Bigram scoring function (p_kneser_ney or bigram_llm).

    Returns:
        dict: Same structure as resolve_oov_robust.
    """
    current_word_oov = not vocab_dict.get(current_word, False)
    next_word_oov    = not vocab_dict.get(next_word, False)

    if current_word_oov and next_word is None:
        return {'current_word_oov': fuzzy_match(current_word, top_fuzz)[:top_n], 'next_word_oov': [], 'type': 'fuzzy-match'}

    elif current_word_oov and next_word_oov:
        return {'current_word_oov': fuzzy_match(current_word, top_fuzz)[:top_n], 'next_word_oov': fuzzy_match(next_word, top_fuzz)[:top_n], 'type': 'fuzzy-match'}

    elif not current_word_oov and next_word_oov:
        # Forward context: rank candidates by both fuzzy similarity and forward bigram LM
        candidates = fuzzy_match(next_word, top_fuzz)
        kn_scores  = [(w, bigram_func(w, current_word)) for w, _ in candidates]
        fuzzy_rank = {w: rank for rank, (w, _) in enumerate(candidates)}
        kn_rank    = {w: rank for rank, (w, _) in enumerate(sorted(kn_scores, key=lambda x: x[1], reverse=True))}
        rrf_scored = [(w, 1/(k + fuzzy_rank[w]) + 1/(k + kn_rank[w])) for w, _ in candidates]
        rrf_scored.sort(key=lambda x: x[1], reverse=True)
        return {'current_word_oov': [], 'next_word_oov': rrf_scored[:top_n], 'type': 'rrf-forward'}

    elif current_word_oov and not next_word_oov:
        # Reverse context: rank candidates by fuzzy similarity and reverse bigram LM
        candidates = fuzzy_match(current_word, top_fuzz)
        kn_scores  = [(w, bigram_func(w, next_word, reverse=True)) for w, _ in candidates]
        fuzzy_rank = {w: rank for rank, (w, _) in enumerate(candidates)}
        kn_rank    = {w: rank for rank, (w, _) in enumerate(sorted(kn_scores, key=lambda x: x[1], reverse=True))}
        rrf_scored = [(w, 1/(k + fuzzy_rank[w]) + 1/(k + kn_rank[w])) for w, _ in candidates]
        rrf_scored.sort(key=lambda x: x[1], reverse=True)
        return {'current_word_oov': rrf_scored[:top_n], 'next_word_oov': [], 'type': 'rrf-reverse'}

    else:
        return {'current_word_oov': [], 'next_word_oov': [], 'type': 'empty'}

## 14. OOV Resolver Unit Tests

Test both resolvers against a comprehensive set of cases covering: both words in-vocab, single OOV with left/right context, both OOV, end-of-sentence (None), single/double character errors, and consecutive OOV tokens.

In [ ]:
test_cases = [
    # both in vocab - should return empty lists
    ("حکومت", "نے"),
    
    # current_word OOV, next_word in vocab - reverse bigram
    ("پاکستاں", "کہا"),
    ("حکومتت", "نے"),
    
    # current_word in vocab, next_word OOV - forward bigram
    ("نے", "کہاا"),
    ("اور", "بھیی"),
    ("یہ", "باات"),
    
    # both OOV - pure fuzzy on both
    ("پاکستاں", "کہاا"),
    ("حکومتت", "بھیی"),
    
    # next_word is None - current word OOV, pure fuzzy fallback
    ("پاکستاں", None),
    ("کہاا", None),
    # single char errors
    ("نے", "کہاا"),
    ("یہ", "باات"),
    ("اور", "بھیی"),
    ("وہ", "آیاا"),
    ("مجھے", "معلووم"),
    ("ہم", "جاائیں"),
    ("آپ", "کہاں"),
    ("کیا", "ہووا"),
    ("انہوں", "نےے"),
    ("پھر", "بھیی"),
    
    # two char errors
    ("پاکستاں", "کہا"),
    ("حکومتت", "نے"),
    ("عوامم", "نے"),
    ("صوررت", "حال"),
    ("پارلیمانن", "نے"),
    
    # both OOV
    ("پاکستاں", "کہاا"),
    ("حکومتت", "بھیی"),
    ("عوامم", "چاہتیی"),
    
    # None cases
    ("پاکستاں", None),
    ("حکومتت", None),
    ("کہاا", None),
    ("آیاا", None)
]

for current, nxt in test_cases:
    result_robust = resolve_oov_robust(current, nxt, 5, 20, 0.98, bigram_func=p_kneser_ney)
    result_rrf    = resolve_oov_rrf(current, nxt, 5, 20,60, bigram_func=p_kneser_ney)
    
    print(f"current: {str(current):12s} | next: {str(nxt):12s}")
    print(f"  [robust] current_word_oov: {result_robust['current_word_oov'][:3]}")
    print(f"  [robust] next_word_oov:    {result_robust['next_word_oov'][:3]}")
    print(f"  [robust] type:             {result_robust['type']}")
    print(f"  [rrf]    current_word_oov: {result_rrf['current_word_oov'][:3]}")
    print(f"  [rrf]    next_word_oov:    {result_rrf['next_word_oov'][:3]}")
    print(f"  [rrf]    type:             {result_rrf['type']}")
    print()

In [ ]:
# Quick vocab membership check for a known misspelling
print('پاکستاں' not in vocab_list)

## 15. Sentence-Level Correction

Apply OOV correction to a full sentence using a **sliding bigram window**:

1. Correct the first token using the second as forward context
2. Slide a window `(tokens[i], tokens[i+1])` across positions 1 to n-2
3. Correct the last token using the second-to-last as reverse context

The `use_rrf` flag switches between the weighted interpolation and RRF strategies.

In [ ]:

urdu_sentences = [
    # ں vs ن
    "پاکستاں کی حکومت نے عوام کو بتایا کہ معیشت بہتر ہو رہی ہے",
    "وزیراعظم نے کہا کہ ملک میں امن و اماں کی صورتحال بہتر ہے",
    
    # double characters
    "سپریم کورٹ نے حکومت کو نوٹس جاری کر دیاا",
    "وزیراعظم نے کہاا کہ ملک میں امن و امان کی صورتحال بہتر ہے",
    "کراچی میں آج شدیید بارش ہوئی اور کئی علاقے زیر آب آ گئے",
    
    # ی vs ے
    "پاکستان اور بھارت کے درمیان کشیدگی میں اضافہ ہو گیی",
    "سپریم کورٹ نے حکومت کو نوٹس جاری کر دیے",
    
    # missing or wrong نقطے
    "پاکستان کی حکومت نے عوام کو ہتایا کہ معیشت بہتر ہو رہی ہے",
    "وزیراعظم نے کہا کہ ملک میں امن و امان کی صورتحال بہتر ہے",
    
    # swapped characters
    "کارچی میں آج شدید بارش ہوئی اور کئی علاقے زیر آب آ گئے",
    "پاکستان روا بھارت کے درمیان کشیدگی میں اضافہ ہو گیا",

    # rare formal/political vocabulary
    "پاکستان کی پارلیمنٹ نے آئینی ترمیم کی منظوری دے دی",
    "عدالت نے ملزم کو بری کرتے ہوئے رہائی کا حکم دیا",
    "وفاقی کابینہ نے اقتصادی پالیسی پر غور کیا",
    
    # proper nouns and place names
    "اسلام آباد میں سفارتخانے کے باہر مظاہرہ ہوا",
    "بلوچستان میں زلزلے کے جھٹکے محسوس کیے گئے",
    
    # technical/medical terms
    "ڈاکٹر نے مریض کو اینٹی بائیوٹک دوائی تجویز کی",
    "حکومت نے ویکسینیشن مہم تیز کرنے کا اعلان کیا",
    "پاکستان کی حکومت نے عوام کو بتایا کہ معیشت بہتر ہو رہی ہے",
    "وزیراعظم نے کہا کہ ملک میں امن و امان کی صورتحال بہتر ہے",
    "کراچی میں آج شدید بارش ہوئی اور کئی علاقے زیر آب آ گئے",
    "سپریم کورٹ نے حکومت کو نوٹس جاری کر دیا",
    "پاکستان اور بھارت کے درمیان کشیدگی میں اضافہ ہو گیا",
        # single OOV mid sentence
    "پاکستان کی حکومتت نے عوام کو بتایا کہ معیشت بہتر ہو رہی ہے",
    "وزیراعظم نے کہاا کہ ملک میں امن و امان کی صورتحال بہتر ہے",
    "کراچی میں آج شدید بارش ہوئی اور کئی علاقے زیر آاب آ گئے",
    
    # OOV at start
    "پاکستاں کی حکومت نے عوام کو بتایا کہ معیشت بہتر ہو رہی ہے",
    "وزیراعظمم نے کہا کہ ملک میں امن و امان کی صورتحال بہتر ہے",
    
    # OOV at end
    "سپریم کورٹ نے حکومت کو نوٹس جاری کر دیاا",
    "پاکستان اور بھارت کے درمیان کشیدگی میں اضافہ ہو گیاا",
    
    # multiple OOV
    "پاکستاں کی حکومتت نے عوامم کو بتایا کہ معیشت بہتر ہو رہی ہے",
    "کراچی میں آج شدیید بارش ہوئی اور کئی علاقے زیر آب آ گئے",
    
    # consecutive OOV
    "سپریم کورٹ نے حکومتت نوٹسس جاری کر دیا",
    "پاکستان اور بھارت کے درمیاان کشیدگیی میں اضافہ ہو گیا",
]
def correct_urdu_sentence(sentence, top_fuzz=20, use_rrf=False):
    tokens = sentence.split()
    
    if len(tokens) == 0:
        return sentence
    if len(tokens) == 1:
        if tokens[0] not in vocab_list:
            candidates = fuzzy_match(tokens[0], top_fuzz)
            return candidates[0][0] if candidates else tokens[0]
        return sentence
    
    corrected = tokens.copy()
    resolve_fn = resolve_oov_rrf if use_rrf else resolve_oov_robust
    
    # handle first word with second as context
    result = resolve_fn(tokens[0], tokens[1])
    if result['current_word_oov']:
        corrected[0] = result['current_word_oov'][0][0]
    
    # slide window across rest of sentence
    for i in range(1, len(tokens) - 1):
        result = resolve_fn(tokens[i], tokens[i + 1])
        if result['current_word_oov']:
            corrected[i] = result['current_word_oov'][0][0]
        if result['next_word_oov']:
            corrected[i + 1] = result['next_word_oov'][0][0]
    
    # handle last word with second to last as context
    result = resolve_fn(tokens[-2], tokens[-1])
    if result['next_word_oov']:
        corrected[-1] = result['next_word_oov'][0][0]
    
    return ' '.join(corrected)


# test it out
for sentence in urdu_sentences:
    print(f"original:  {sentence}")
    print(f"robust:    {correct_urdu_sentence(sentence, use_rrf=False)}")
    print(f"rrf:       {correct_urdu_sentence(sentence, use_rrf=True)}")
    print()

## 16. WER and CER Metrics

### Word Error Rate (WER)

$$\text{WER} = \frac{\text{lev}(\text{ref}_{words}, \text{hyp}_{words})}{|\text{ref}_{words}|}$$

Computed by treating the sentence as a sequence of word tokens and applying Levenshtein distance at the word level.

### Character Error Rate (CER) — SCLITE-style

Uses SCLITE-style word-level alignment first, then counts character-level errors within each aligned word pair:

1. Align reference and hypothesis at the **word level** using the Levenshtein DP matrix (backtracking diagonally first to prefer substitutions over indels)
2. For each aligned pair:
   - **Match** → add `len(ref_word)` to total characters
   - **Deletion** → add `len(ref_word)` to both errors and total
   - **Insertion** → add `len(hyp_word)` to errors only
   - **Substitution** → add character-level `levenshtein(ref_word, hyp_word)` to errors; add `len(ref_word)` to total

$$\text{CER} = \frac{\sum \text{char errors}}{\sum |\text{ref word chars}|}$$

In [ ]:
def sclite_align(ref, hyp):
    """
    Align reference and hypothesis word sequences using Levenshtein backtracking.

    Follows SCLITE convention: diagonal (match/substitution) is preferred over
    vertical (deletion) or horizontal (insertion) to minimize indel operations.

    Args:
        ref (list[str]): Reference word tokens.
        hyp (list[str]): Hypothesis word tokens.

    Returns:
        list[tuple]: Alignment pairs (ref_word, hyp_word).
                     None in ref position = insertion; None in hyp = deletion.
    """
    dp = levenshtein(ref, hyp, dp_matrix=True)
    i, j = len(ref), len(hyp)
    aligned = []

    while i > 0 or j > 0:
        if i > 0 and j > 0:
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            if dp[i][j] == dp[i-1][j-1] + cost:   # diagonal: match or substitution
                aligned.append((ref[i-1], hyp[j-1]))
                i -= 1; j -= 1
            elif dp[i][j] == dp[i-1][j] + 1:        # vertical: deletion
                aligned.append((ref[i-1], None))
                i -= 1
            else:                                    # horizontal: insertion
                aligned.append((None, hyp[j-1]))
                j -= 1
        elif i > 0:
            aligned.append((ref[i-1], None)); i -= 1
        else:
            aligned.append((None, hyp[j-1])); j -= 1

    return aligned[::-1]


def sclite_cer(ref_sentence, hyp_sentence):
    """
    Compute SCLITE-style Character Error Rate (CER).

    Aligns at word level first, then measures character-level errors
    within each aligned word pair.

    Args:
        ref_sentence (str): Reference sentence string.
        hyp_sentence (str): Hypothesis sentence string.

    Returns:
        float: CER in [0, 1]. Returns 0.0 if reference is empty.
    """
    aligned = sclite_align(ref_sentence.split(), hyp_sentence.split())
    errors, total = 0, 0

    for r, h in aligned:
        if r is None:          errors += len(h)                        # insertion
        elif h is None:        errors += len(r); total += len(r)       # deletion
        elif r == h:           total += len(r)                         # match
        else:                  errors += levenshtein(r, h); total += len(r)  # substitution

    return errors / total if total else 0.0


def wer(ref_sentence, hyp_sentence):
    """
    Compute Word Error Rate (WER).

    Args:
        ref_sentence (str): Reference sentence string.
        hyp_sentence (str): Hypothesis sentence string.

    Returns:
        float: WER. Returns 0.0 if both are identical.
    """
    ref, hyp = ref_sentence.split(), hyp_sentence.split()
    return levenshtein(ref, hyp) / len(ref)

In [ ]:
# Unit tests for WER and CER across a range of edit types
test_cases = [
    ("the cat sat on the mat", "the cat sat on the mat"),     # perfect → WER=0, CER=0
    ("the cat sat on the mat", "the cat sat on a mat"),       # 1 sub → WER=1/6
    ("the cat sat on the mat", "cat sat on the mat"),         # 1 del → WER=1/6
    ("the cat sat on the mat", "the the cat sat on the mat"), # 1 ins → WER=1/6
    ("hello world", ""),                                      # all del → WER=1.0
    ("", "hello world"),                                      # all ins → WER=inf/edge
    ("john ate the apple", "john eight the apple"),           # 1 phonetic sub
    ("we are going home", "we going home"),                   # 1 del mid-sentence
    ("simple test", "simpel tset"),                          # char transpositions
    ("a b c d e", "e d c b a"),                              # full reversal → WER=1.0
    ("hello world foo", "hello foo"),                        # word deletion (CER charges len('world')=5)
    ("a big red ball", "a ball"),                            # 2 consecutive deletions
    ("the quick fox", "quick fox"),                          # deletion at start
    ("go home now", "go home"),                              # deletion at end
    ("one two three four", "two"),                           # 3 words deleted
    ("a b c d e f", "a c e"),                                # alternating deletions
    ("cat sat mat", "cat mat"),                              # deletion vs substitution ambiguity
    ("پاکستان کی حکومت", ""),                                 # full Urdu sentence deleted
]

for ref, hyp in test_cases:
    w = wer(ref, hyp) if ref else float('inf')
    c = sclite_cer(ref, hyp) if ref else float('inf')
    print(f"REF: {ref[:30]:30s} | HYP: {hyp[:30]:30s} | WER: {w:.2f} | CER: {c:.2f}")

## 17. Apply OOV Correction to All Models

Apply both correction strategies (`robust` and `rrf`) to the normalized predictions of all four models. Each model gains two new columns: `{model}_norm_robust` and `{model}_norm_rrf`.

In [ ]:
# Derive model names directly from df columns — stays in sync regardless of
# what the source TSV filenames are. Excludes path, ground_truth, and any
# columns already created by the normalization or correction steps.
models = [
    col for col in df.columns
    if col not in ['path', 'ground_truth']
    and not col.endswith('_norm')
    and not col.endswith('_robust')
    and not col.endswith('_rrf')
]
print("Models found:", models)

for model in models:
    norm_col = f'{model}_norm'
    # Apply weighted interpolation correction
    df[f'{model}_norm_robust'] = df[norm_col].apply(
        lambda s: correct_urdu_sentence(s, use_rrf=False)
    )
    # Apply Reciprocal Rank Fusion correction
    df[f'{model}_norm_rrf'] = df[norm_col].apply(
        lambda s: correct_urdu_sentence(s, use_rrf=True)
    )

## 18. Evaluation: WER and CER Across All Stages

Score each model across four pipeline stages:

| Category | Description |
|---|---|
| `untouched` | Raw ASR output vs raw ground truth |
| `norm_baseline` | Normalized output vs normalized ground truth |
| `robust` | Robust-corrected output vs normalized ground truth |
| `rrf` | RRF-corrected output vs normalized ground truth |

In [ ]:
def score_col(ref_col, hyp_col):
    """Compute mean WER and CER across all rows for a given ref/hyp column pair."""
    wer_scores = df.apply(lambda row: wer(row[ref_col], row[hyp_col]), axis=1)
    cer_scores = df.apply(lambda row: sclite_cer(row[ref_col], row[hyp_col]), axis=1)
    return wer_scores.mean(), cer_scores.mean()

# Re-derive model names here so this cell can run independently without
# relying on the 'models' variable from the correction cell above
models = [
    col for col in df.columns
    if col not in ['path', 'ground_truth']
    and not col.endswith('_norm')
    and not col.endswith('_robust')
    and not col.endswith('_rrf')
]

results = []

for model in models:
    # Stage 1: raw output vs raw ground truth
    w, c = score_col('ground_truth', model)
    results.append({'model': model, 'category': 'untouched', 'WER': w, 'CER': c})

    # Stage 2: normalized output vs normalized ground truth (punctuation/script-neutral)
    w, c = score_col('ground_truth_norm', f'{model}_norm')
    results.append({'model': model, 'category': 'norm_baseline', 'WER': w, 'CER': c})

    # Stage 3: robust OOV-corrected vs normalized ground truth
    w, c = score_col('ground_truth_norm', f'{model}_norm_robust')
    results.append({'model': model, 'category': 'robust', 'WER': w, 'CER': c})

    # Stage 4: RRF OOV-corrected vs normalized ground truth
    w, c = score_col('ground_truth_norm', f'{model}_norm_rrf')
    results.append({'model': model, 'category': 'rrf', 'WER': w, 'CER': c})

results_df = pd.DataFrame(results)

In [ ]:
# Pivot results into separate WER and CER tables for easy comparison
# Columns ordered: untouched → norm_baseline → robust → rrf
wer_table = results_df.pivot(index='model', columns='category', values='WER')[['untouched', 'norm_baseline', 'robust', 'rrf']]
cer_table = results_df.pivot(index='model', columns='category', values='CER')[['untouched', 'norm_baseline', 'robust', 'rrf']]

wer_table.columns.name = None
cer_table.columns.name = None

print("=== WER ===")
print(wer_table.round(4).to_string())
print("\n=== CER ===")
print(cer_table.round(4).to_string())

## 19. Save Outputs

Persist the final corrected ensemble and the evaluation results to disk.

In [ ]:
# Save the corrected ensemble (all normalization and OOV correction columns included)
df.to_csv(f"{ASR_ENSEMBLE_DIR}/ensemble_corrected.tsv", sep='\t', index=False)

# Save the WER/CER evaluation table for all models and pipeline stages
results_df.to_csv(f"{ASR_EVALUATION_DIR}/ensemble_correction_eval.tsv", sep='\t', index=False)